# Regras e preparação por grupo

Este notebook aplica as granularidades, sementes e divisões temporais registradas na configuração. Ele não cria features nem altera os arquivos brutos.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs' / 'base_experiment_rules.json').exists()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from discovery_series_temporais.experiment_config import load_experiment_rules
from discovery_series_temporais.series import build_origin_schedule, load_candidate_target

print(f'Raiz do projeto: {ROOT}')

Raiz do projeto: C:\Users\mayumishimizu-ieg\.codex\.chatgpt-projects\g-p-6ad36ceb09bc81918ea40186a7712f30\DiscoverySeriesTemporais


In [2]:
rules = load_experiment_rules(ROOT / 'configs' / 'base_experiment_rules.json')
configured = pd.DataFrame([rule.__dict__ for rule in rules.values()])
display(configured[['base_id', 'group', 'granularity', 'train_fraction', 'test_fraction', 'random_state', 'target_rule']])
assert all((configured['train_fraction'] + configured['test_fraction']).round(12).eq(1.0))
print('Todas as frações de treino e teste somam 100%.')

,base_id,group,granularity,train_fraction,test_fraction,random_state,target_rule
0,1,Grupo 1,daily,0.70,0.30,42,Close at daily cadence
1,2,Grupo 2,hourly,0.80,0.20,67,One traffic_volume per clock hour; repeated ta...
2,3,Grupo 3,hourly,0.80,0.20,42,PM2.5 at hourly cadence; unobserved target val...
3,4,Grupo 4,hourly,0.80,0.20,42,Hourly mean of valid ten-minute T (degC) readi...
4,5,Grupo 5,weekly ending Friday,0.75,0.25,42,Last observed GOLD_PRICE in each Friday-ended ...


Todas as frações de treino e teste somam 100%.


In [3]:
prepared = []
for base_id in range(1, 6):
    candidate = load_candidate_target(base_id, ROOT / 'data' / 'processed', ROOT / 'configs' / 'base_experiment_rules.json')
    schedule = build_origin_schedule(candidate)
    prepared.append({
        'base': base_id,
        'frequency': candidate.frequency_label,
        'rows_after_granularity_rule': len(candidate.data),
        'missing_target': int(candidate.data['target'].isna().sum()),
        'test_origins_pilot_h1': len(schedule),
        'first_test_origin': schedule['origin_timestamp'].min(),
        'random_state': candidate.random_state,
    })

prepared_summary = pd.DataFrame(prepared)
display(prepared_summary)
print('A agenda usa h=1 somente como piloto. O horizonte final continua pendente.')

,base,frequency,rows_after_granularity_rule,missing_target,test_origins_pilot_h1,first_test_origin,random_state
0,1,daily,2836,0,851,2022-06-08 00:00:00,42
1,2,hourly (target-only consolidation),40575,0,7598,2017-11-16 20:00:00,67
2,3,hourly,35064,925,6786,2016-05-15 21:00:00,42
3,4,hourly mean,70129,88,14008,2015-05-25 06:00:00,42
4,5,weekly ending Friday using last observed price,2398,254,481,2002-10-11 00:00:00,42


A agenda usa h=1 somente como piloto. O horizonte final continua pendente.
